<a href="https://colab.research.google.com/github/charre2021/reasoning_agent_example/blob/main/reasoning_agent_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from typing import List, Dict, Any
import random
from openai import OpenAI
from google.colab import userdata

class LLM:
    def __init__(self, model : str, api_key : str) -> None:
        self.client = OpenAI(api_key = api_key)
        self.model = model

    def generate(self, prompt : str) -> str:
        response = self.client.responses.create(
            model = self.model,
            input = prompt
        )
        return response.output_text

class HierarchicalGoal:
    def __init__(self, parent_goal: str) -> None:
        self.parent_goal = parent_goal
        self.sub_goals : Dict[int,'HierarchicalGoal'] = {}
        self.completed = False

    def need_sub_goals(self, additional_context : str, llm : LLM) -> bool:
        while True:
            additional_context = f"Goal: {self.parent_goal}\n"
            additional_context += "Recent observations:\n"
            additional_context += """\n
            Think about the current situation and the goal. Does this goal have subgoals?
            If yes, say, 'True.' If no, say, 'False.' Limit your response to one word.
            """
            try:
                response = bool(llm.generate(additional_context))
                break;
            except TypeError:
                additional_context += """The format of your response was incorrect. Let's try that again.
                Remember to limit your response to just 'True' or 'False'."""
                continue
        return response

    def generate_sub_goals(self, additional_context : str, llm : LLM) -> List[str]:
        while True:
            SUBGOAL_DELIMITER = "NEXT SUBGOAL:"
            additional_context += "You said that subgoals were needed for this goal."
            additional_context += f"Ok, can you please list all subgoals? Delimit each with '{SUBGOAL_DELIMITER}'."
            try:
                response = llm.generate(additional_context).split(f"{SUBGOAL_DELIMITER}")
                break;
            except:
                additional_context += f"""The format of your response was incorrect. Let's try that again.
                Remember to list all subgoals with the '{SUBGOAL_DELIMITER}'."""
                continue
        return response

    def add_sub_goal(self, sub_goal : str) -> None:
        if hash(sub_goal) not in self.sub_goals:
            sg = HierarchicalGoal(sub_goal)
            self.sub_goals[hash(sub_goal)] = sg

    def mark_completed(self) -> None:
        self.completed = True

class LLMAgent:
    def __init__(self, llm, action_space: List[str]) -> None:
        self.llm = llm
        self.action_space = action_space
        self.memory = []
        self.goal_stack = []
        self.current_goal = None

    def add_goal(self, goal: str) -> None:
        current_root_goal = HierarchicalGoal(goal)
        self.goal_stack.append(current_root_goal)
        are_sub_goals_needed = current_root_goal.need_sub_goals(self.memory[-5:], self.llm)
        if are_sub_goals_needed:
            sub_goals_list =
            for sub_goal in sub_goals_list:
                current_root_goal.add_sub_goal(sub_goal)
                self.add_goal(sub_goal)
        else:
            self.current_goal = self.goal_stack.pop(-1)

    def perceive(self, observation: str) -> None:
        self.memory.append(observation)

    def think(self) -> str:
        context = f"Goal: {self.current_goal}\n"
        context += "Recent observations:\n"
        context += "\n".join(self.memory[-5:])
        context += "\nThink about the current situation and the goal. What should be done next?"

        return self.llm.generate(context)

    def decide(self, thought: str) -> str:
        context = f"Thought: {thought}\n"
        context += "Based on this thought, choose the most appropriate action from the following:\n"
        context += ", ".join(self.action_space)
        context += "\nChosen action:"

        return self.llm.generate(context)


    def act(self, action: str) -> str:
        outcomes = [
            f"Action '{action}' was successful.",
            f"Action '{action}' failed.",
            f"Action '{action}' had an unexpected outcome."
        ]
        return random.choice(outcomes)

    def run_step(self):
        thought = self.think()
        action = self.decide(thought)
        outcome = self.act(action)
        self.perceive(outcome)
        return thought, action, outcome



In [ ]:
llm = LLM("o4-mini", userdata.get("openai"))
action_space = ["move", "grab", "drop", "use", "talk"]
agent = LLMAgent(llm, action_space)

agent.add_goal("Find the key and unlock the door.")
agent.perceive("You are in a room with a table and a chair. There's a drawer in the table.")

for _ in range(5):
    thought, action, outcome = agent.run_step()
    print(f"Thought: {thought}")
    print(f"Action: {action}")
    print(f"Outcome: {outcome}")